In [ ]:
import fitz 
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\Ronish\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"# PyMuPDF
import os
from PIL import Image
import io
import glob
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from transformers import pipeline


# Load Pdf


In [2]:
import os
import glob
import fitz  # PyMuPDF

PDF_FOLDER = "data/"
pdf_paths = glob.glob(os.path.join(PDF_FOLDER, "*.pdf"))
all_text = []

for pdf_path in pdf_paths:
    print(f"\nProcessing: {os.path.basename(pdf_path)}")
    doc = fitz.open(pdf_path)

    for page_num, page in enumerate(doc):
        text = page.get_text()

        if text and text.strip():
            all_text.append({
                "text": text,
                "source": os.path.basename(pdf_path),
                "page": page_num + 1
            })

    doc.close()
    print(f"  → Done: {os.path.basename(pdf_path)}")

print(f"\nTotal pages extracted: {len(all_text)}")


Processing: Chapter_6_Machine Learning.pdf
  → Done: Chapter_6_Machine Learning.pdf

Processing: COCOMO.pdf
  → Done: COCOMO.pdf

Processing: sample.pdf
  → Done: sample.pdf

Total pages extracted: 164


In [3]:
documents = []
for item in all_text:
    documents.append(
        Document(
            page_content=item["text"],
            metadata={
                "source": item["source"],
                "page": item["page"]
            }
        )
    )

# Split into Chunks

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print("Sample metadata:", chunks[0].metadata)

Total chunks: 190
Sample metadata: {'source': 'Chapter_6_Machine Learning.pdf', 'page': 1}


# Create Embedding

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Ronish\AppData\Local\Temp\ipykernel_13508\2671871813.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Create Faiss Vector DB


In [6]:
vectorstore = FAISS.from_documents(chunks, embeddings)


In [7]:
query = "what isscript?"

docs = vectorstore.similarity_search(query)

for d in docs:
    print("SOURCE:", d.metadata.get("source", "N/A"))
    print("PAGE:", d.metadata.get("page", "N/A"))
    print(d.page_content[:200])
    print("-----")

SOURCE: sample.pdf
PAGE: 32
Scripts are a development of the idea of 
representing actions and events using semantic 
networks. 
With scripts the idea is that whole set of actions 
fall into stereotypical (similar tune) patter
-----
SOURCE: COCOMO.pdf
PAGE: 4
7/30/2025
Prepared by: Er. Sumit Sharma
4
4. Module Code and Test
❑Write the actual code for each part as planned 
in the detailed design.
❑Build the functions, logic, and connections.
3. Detailed Des
-----
SOURCE: COCOMO.pdf
PAGE: 1
Software 
Engineering
BCT(III/I)
7/30/2025
1
COCOMO
-----
SOURCE: sample.pdf
PAGE: 12
Different types of KR techniques are:
Semantic Network
Frames
Conceptual Dependencies
Scripts
-----


In [ ]:
import os



# 1. Setup Groq
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Environment variable 'GROQ_API_KEY' is not set. Add it to your environment or .env file.")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY
)

# 2. Setup retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 15})

# 3. Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based on the context below.

Context: {context}

Question: {question}
""")

# 4. Chain
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask a question!
answer = chain.invoke('what is script learning? explain in detail')
print(answer)

# ADDED
print("\nSources used:")
for d in docs:
    print(f"  - {d.metadata.get('source')} | Page {d.metadata.get('page')}")


Based on the provided context, script learning is a concept that involves representing actions and events using semantic networks. It is an extension of the idea of semantic networks, where a whole set of actions fall into stereotypical patterns.

Scripts make use of the idea of defaults, with a class defining the roles in some action, and individual instances of the class having the roles filled in. This allows for the creation of a framework that can be used to analyze and understand complex events and actions.

The concept of script learning is based on the idea that many actions and events follow predictable patterns, and that by identifying these patterns, it is possible to create a script that can be used to understand and analyze similar events. This script can be thought of as a template or a framework that outlines the typical sequence of actions and events that occur in a particular situation.

Script learning involves the use of semantic networks to represent the relationshi

In [9]:
vectorstore.save_local("faiss_index")
print("Vectorstore saved!")

Vectorstore saved!
